# Retrievals: Advanced features

Written by [Evert Nasedkin](mailto:nasedkinevert@gmail.com?subject=[petitRADTRANS]%20Retrievals).
Please cite pRT's retrieval package [(Nasedkin et al. 2024)](https://ui.adsabs.harvard.edu/abs/2024JOSS....9.5875N/abstract) in addition to pRT [(Mollière et al. 2019)](https://ui.adsabs.harvard.edu/abs/2019A%26A...627A..67M/abstract) if you make use of the retrieval package for your work.

Please make sure you thoroughly understand the retrieval setups described in the ["Basic Retrieval Tutorial"](retrieval_basic.html), the ["multiple data sets tutorial"](retrieval_multiple_data.html) and the ["emission spectrum retrieval tutorial"](retrieval_emission.html) before starting on this notebook. 

In this tutorial we will present some of the more advanced features of the retrieval package. Some of these are more fragile than the standard retrieval features, and may break unexpectedly. You should be comfortable writing your own model generating function for the retrieval, and be willing to do some debugging if something goes wrong. Caveats and warnings out of the way, let's get started. 

## Getting started
We'll start off using a similar set up to the ["emission spectrum retrieval tutorial"](retrieval_emission.html).

In [1]:
# Let's start by importing everything we need.
import os
# To not have numpy start parallelizing on its own
os.environ["OMP_NUM_THREADS"] = "1"

import numpy as np
import matplotlib.pyplot as plt

import petitRADTRANS as prt
from petitRADTRANS import physical_constants as cst

# Import the class used to set up the retrieval.
from petitRADTRANS.retrieval import Retrieval,RetrievalConfig
# Import Prior functions, if necessary.
from petitRADTRANS.retrieval.utils import gaussian_prior
# Import atmospheric model function
from petitRADTRANS.retrieval.models import gradient_profile_emission

In [2]:
# Define the pRT run setup
retrieval_config = RetrievalConfig(
    retrieval_name="HR8799e_example", # give a useful name for your retrieval
    run_mode="retrieval",  # 'retrieve' to run, or 'evaluate' to make plots
    adaptive_mesh_refinement=True,  # adaptive mesh refinement, slower if True
    scattering_in_emission=True  #  add scattering for emission spectra clouds
)

For this example we include the GRAVITY data as published in Mollière et al. (2020). To reproduce the published results, please also include the archival SPHERE and GPI data from [Zurlo et al. (2015)](https://www.aanda.org/articles/aa/full_html/2016/03/aa26835-15/aa26835-15.html) and [Greenbaum et al. (2016)](https://iopscience.iop.org/article/10.3847/1538-3881/aabcb8).

In [ ]:
# Read in Data

# Here we import petitRADTRANS to find the path of the example files on your machine
# In general this is not required, you just put the files in the folder that you are running
# Your script in, for example
import petitRADTRANS # need to get the name for the example data
path_to_data = "./"

retrieval_config.add_data(
    name='GRAVITY',
    path_to_observations=f"{path_to_data}/retrievals/emission/observations/HR8799e_Spectra.fits",
    data_resolution=500,
    model_resolution=1000,
    model_generating_function = gradient_profile_emission,
    system_distance=41.29 * cst.pc,  # distance to system in cm
    scale_flux=True, # let the retrieval scale the flux by a multiplicative factor
    scale_uncertainties=True, # let the retrieval scale the flux uncertainties by a multiplicative factor
    fit_flux_offset=False, # Don't use an additive offset
    subtract_continuum=False, # Don't use continuum subtraction
    radvel=False, # Don't apply radial velocity shift
    line_opacity_mode='c-k',
    radtrans_object=None, # if we wanted, we could direcly pass an existing Radtrans object here
    wavelengths=None, # likewise, rather than loading the data from file we could manually load it

## Photometric data

If we want to add photometry, we can do that as well! The photometry file should have the format:

`# Name, lower wavelength edge [um], upper wavelength edge [um], flux density [W/m2/micron], flux error [W/m2/micron]`

You are required to provide a model function for calculating the spectrum, as with spectral data, but also a photometric transformation function, which is used to convert the model spectrum into synthetic photometry. This would typically make use of the transmission function for a particular filter. We recommend the use of the `species` package (https://species.readthedocs.io/), in particular the `SyntheticPhotometry` module to provide these functions. If no function is provided, the RetrievalConfig will attempt to import `species` to use this module, using the `name` provided as the filter name.

If you are using transmission spectra, your photometric transformation function should model the difference between the clear and occulted stellar spectrum, returning the difference in (planet radius/stellar radius)^2.

In [4]:
retrieval_config.add_photometry(
    path_to_data + 'retrievals/emission/observations/hr8799e_photometry.txt',
    gradient_profile_emission,
    model_resolution=40,
    distance=41.29 * cst.pc,
    scale_flux=False,
    wlen_range_micron=None,
    photometric_transformation_function=None, # define the function to transform model spectrum to photometric points
    )
# The photometric transformation function should be defined for each photometric band, which means we might need to
# call this function several times with different photometry files if you don't want to use the default functions,
# which are downloaded from the SVO Filter Profile Service using species.

species

Version: 0.9.1.dev90+g6857078.d20250807
Working folder: /Users/nasedkin/python-packages/petitRADTRANS/docs/content/notebooks

Configuration file: /Users/nasedkin/python-packages/petitRADTRANS/docs/content/notebooks/species_config.ini
Database file: species_database.hdf5
Data folder: data

Configuration settings:
   - Database: species_database.hdf5
   - Data folder: data
   - Magnitude of Vega: 0.03

Multiprocessing: mpi4py installed
Process number 1 out of 1...


## Parameters and Priors

Here we add all of the parameters used in the retrieval for HR 8799 e, following the prescription of Mollière 2020.
There are many other approaches we could take: varying the temperature structure parameterisation, retrieving different cloud properties, adding in a blackbody CPD and so on.
It is highly recommended to look at the API documentation of `models.py`, or at the [Retrieval Models Tutorial](retrieval_models.html) to get a better idea of what options are available.

In [ ]:
# Add parameters, and priors for free parameters

# This run uses the model of Molliere (2020) for HR8799e
# The lambda function provide uniform priors

# Distance to the planet in cm
retrieval_config.add_parameter(
    name='system_distance',
    is_free_parameter=False,
    value=41.2925 * cst.pc
)

# Log of the surface gravity in cgs units.
retrieval_config.add_parameter(
    'log_g',
    True,
    transform_prior_cube_coordinate = lambda x : 2 + 3.5 * x
)

# Planet radius in cm
retrieval_config.add_parameter(
    'planet_radius',
    True,
    transform_prior_cube_coordinate=lambda x : ( 0.7 + 1.3 * x) * cst.r_jup_mean
)

# Temperature profile, based on Zhang+ 2023
n_layers = 10
retrieval_config.add_parameter('top_of_atmosphere_pressure', False, value = -6)
retrieval_config.add_parameter('N_layers', False, value = n_layers)
retrieval_config.add_parameter('T_bottom', True,
                            transform_prior_cube_coordinate = \
                            lambda x : 2000.0 + 10000.0*x)

#dts = [0.07,0.10,0.18,0.27,0.24,0.25]
retrieval_config.add_parameter(f'PTslope_1',
                            True,
                            transform_prior_cube_coordinate = \
                                lambda x : gaussian_prior(x,0.25,0.1))
retrieval_config.add_parameter(f'PTslope_2',
                            True,
                            transform_prior_cube_coordinate = \
                                lambda x : gaussian_prior(x,0.25,0.1))
retrieval_config.add_parameter(f'PTslope_3',
                            True,
                            transform_prior_cube_coordinate = \
                                lambda x : gaussian_prior(x,0.26,0.1))
retrieval_config.add_parameter(f'PTslope_4',
                            True,
                            transform_prior_cube_coordinate = \
                                lambda x : gaussian_prior(x,0.2,0.1))
retrieval_config.add_parameter(f'PTslope_5',
                            True,
                            transform_prior_cube_coordinate = \
                                lambda x : gaussian_prior(x,0.12,0.1))
retrieval_config.add_parameter(f'PTslope_6',
                            True,
                            transform_prior_cube_coordinate = \
                                lambda x : gaussian_prior(x,0.07,0.1))
retrieval_config.add_parameter(f'PTslope_7',
                            True,
                            transform_prior_cube_coordinate = \
                                lambda x : gaussian_prior(x,0.0,0.5))
retrieval_config.add_parameter(f'PTslope_8',
                            True,
                            transform_prior_cube_coordinate = \
                                lambda x : gaussian_prior(x,0.0,0.5))
retrieval_config.add_parameter(f'PTslope_9',
                            True,
                            transform_prior_cube_coordinate = \
                                lambda x : gaussian_prior(x,0.0,0.05))
retrieval_config.add_parameter(f'PTslope_10',
                            True,
                            transform_prior_cube_coordinate = \
                                lambda x : gaussian_prior(x,0.0,0.05))




## Chemistry
There are multiple ways to parameterise the chemistry in a retrieval: we can rely on equilibrium chemistry, a simple parameterisation of disequilibrium chemistry, we can freely retrieve vertically constant abundances, we can vary the abundance profile with pressure, or we can use some combination of all of the above. Here we'll set up a typical disequilibrium chemistry retrieval, but allow CO to vary as a function of altitude.

In [6]:
# Chemistry
# A 'free retrieval' would have each line species as a parameter
# Using a (dis)equilibrium model, we only supply bulk parameters.
# Carbon quench pressure
retrieval_config.add_parameter(
    'log_pquench',
    True,
    transform_prior_cube_coordinate=lambda x : -6.0 + 9.0 * x
    )
# Metallicity [Fe/H]
retrieval_config.add_parameter(
    'Fe/H',
    True,
    transform_prior_cube_coordinate=lambda x : -1.5 + 3.0 * x
)
# C/O ratio
retrieval_config.add_parameter(
    'C/O',
    True,
    transform_prior_cube_coordinate=lambda x : 0.1+1.5*x
)

In [7]:
retrieval_config.add_pressure_varying_line_species(
    'CO-NatAbund__HITEMP',
    mode='linear', # 'linear' or 'cubic' interpolation between nodes, or 'stepped', where the abundance is constant between nodes
    pressure_spacing='relative', # fit for the location of the nodes in pressure space relative to the previous node.
    n_nodes=3, # How many nodes to use. In addition the abundance will be fit at the top and bottom of the atmosphere.
    free_mass_fraction_limits=(-7.0, 0.0), # Limits of the mass fraction abundance
    log_pressure_range_prior=(0, 9), # Range in which each pressure node can vary relative to the node above.
    )

In [8]:
# Clouds
# Based on an Ackermann-Marley (2001) cloud model
# Width of particle size distribution
retrieval_config.add_parameter(
    'sigma_lnorm',
    True,
    transform_prior_cube_coordinate=lambda x : 1.05 + 1.95 * x
)
# Sedimentation parameter for each cloud species
retrieval_config.add_parameter('fsed_MgSiO3(s)',True,\
                                transform_prior_cube_coordinate = \
                                lambda x : 0.0 + 10.0*x)
retrieval_config.add_parameter('fsed_Fe(s)',True,\
                                transform_prior_cube_coordinate = \
                                lambda x : 0.0 + 10.0*x)

# Cloud particle radius for each cloud species
retrieval_config.add_parameter('log_cloud_radius_MgSiO3(s)',True,\
                            transform_prior_cube_coordinate = \
                            lambda x : -7 + 10.0*x)
retrieval_config.add_parameter('log_cloud_radius_Fe(s)',True,\
                            transform_prior_cube_coordinate = \
                            lambda x : -7 + 10.0*x)

# Cloud coverage fraction. Can be defined for all cloud species, or for a list of species.
retrieval_config.add_parameter('cloud_coverage',True,\
                            transform_prior_cube_coordinate = \
                            lambda x : x)
retrieval_config.add_parameter("remove_cloud_species",False,["MgSiO3(s)_crystalline__DHS"])


In [9]:
# Define opacity species to be included
retrieval_config.set_rayleigh_species(['H2', 'He'])
retrieval_config.set_continuum_opacities(['H2--H2', 'H2--He'])
retrieval_config.set_line_species(
    [
        'H2O__POKAZATEL',
        'CO-NatAbund', # freely retrieving the abundance profile of CO
        'CH4__MM',
        'CO2',
        'HCN',
        'FeH',
        'H2S',
        'NH3',
        'PH3',
        'Na',
        'K',
        'TiO',
        'VO',
        'SiO'
    ],
    eq = False
)



H2O__POKAZATEL H2O__POKAZATEL
CO-NatAbund CO-NatAbund
CH4__MM CH4__MM
CO2 CO2
HCN HCN
FeH FeH
H2S H2S
NH3 NH3
PH3 PH3
Na Na
K K
TiO TiO
VO VO
SiO SiO


## Free cloud parameterisation
Rather than using the condensation curve to define where the clouds are located, we can directly retrieve the cloud base pressure and abundance.

In [10]:

retrieval_config.add_cloud_species('Fe(s)_crystalline__DHS', use_equilibrium_chemistry=False, free_mass_fraction_limits=(-7.0, -1.0), cloud_base_pressure_limits=(-6,3))
retrieval_config.add_cloud_species('MgSiO3(s)_crystalline__DHS', use_equilibrium_chemistry=False, free_mass_fraction_limits=(-7.0, -1.0), cloud_base_pressure_limits=(-6,3))

Fe(s)_crystalline__DHS Fe(s)_crystalline__DHS-NatAbund 56Fe(s)_crystalline Fe(s)
MgSiO3(s)_crystalline__DHS MgSiO3(s)_crystalline__DHS-NatAbund 24Mg-28Si-16O3(s)_crystalline MgSiO3(s)


In [7]:
retrieval = Retrieval(
    retrieval_config,
    output_directory="",
    evaluate_sample_spectra=False
)

Setting up Radtrans object for data 'GRAVITY'...
Setting up AMR pressure grid.
Loading Radtrans opacities...
 Done.ng line opacities of species 'H2O__POKAZATEL.R1000' from file '/home/dblain/petitRADTRANS/input_data/opacities/lines/correlated_k/H2O/1H2-16O/1H2-16O__POKAZATEL.R1000_0.3-50mu.ktable.petitRADTRANS.h5'...
 Loading line opacities of species 'CO-NatAbund.R1000' from file '/home/dblain/petitRADTRANS/input_data/opacities/lines/correlated_k/CO/C-O-NatAbund/C-O-NatAbund__HITEMP.R1000_0.1-250mu.ktable.petitRADTRANS.h5'... Done.
 Done.ng line opacities of species 'CH4.R1000' from file '/home/dblain/petitRADTRANS/input_data/opacities/lines/correlated_k/CH4/12C-1H4/12C-1H4__YT34to10.R1000_0.3-50mu.ktable.petitRADTRANS.h5'...
 Done.ng line opacities of species 'CO2.R1000' from file '/home/dblain/petitRADTRANS/input_data/opacities/lines/correlated_k/CO2/12C-16O2/12C-16O2__UCL-4000.R1000_0.3-50mu.ktable.petitRADTRANS.h5'...
 Loading line opacities of species 'HCN.R1000' from file '/home

In [ ]:
# Before we run the retrieval, let's set up plotting.

for parameter in retrieval_config.parameters:
    retrieval_config.parameters[parameter].plot_in_corner = True

In [9]:
retrieval = Retrieval(
    retrieval_config,
    output_directory="./",
    evaluate_sample_spectra=False
)

Using provided Radtrans object for data 'GRAVITY'...
Using provided Radtrans object for data 'Keck/NIRC2.Ks'...
Using provided Radtrans object for data 'Paranal/NACO.Lp'...
Using provided Radtrans object for data 'Paranal/NACO.NB405'...
Using provided Radtrans object for data 'Paranal/SPHERE.IRDIS_B_J'...
Using provided Radtrans object for data 'Paranal/SPHERE.IRDIS_D_H23_2'...
Using provided Radtrans object for data 'Paranal/SPHERE.IRDIS_D_H23_3'...
Using provided Radtrans object for data 'Paranal/SPHERE.IRDIS_D_K12_1'...
Using provided Radtrans object for data 'Paranal/SPHERE.IRDIS_D_K12_2'...


In [ ]:
# Set up plotting — these attributes live on the RetrievalPlotter after Retrieval is created
# Define axis properties of spectral plot if run_mode == 'evaluate'
retrieval.plotter.spec_xlabel = 'Wavelength [micron]'
retrieval.plotter.spec_ylabel = "Flux [W/m2/micron]"
retrieval.plotter.y_axis_scaling = 1.0
retrieval.plotter.xscale = 'log'
retrieval.plotter.yscale = 'linear'
retrieval.plotter.resolution = 100.0  # maximum resolution, will rebin the data
retrieval.plotter.nsample = 100  # if we want a plot with many sampled spectra

# Define from which observation object to take P-T in evaluation mode, add PT-envelope plotting options
retrieval.plotter.reference_data_name = 'GRAVITY'
retrieval.plotter.temp_limits = [150, 3000]
retrieval.plotter.press_limits = [1e2, 1e-5]

In [ ]:
# Set up plotting — these attributes live on the RetrievalPlotter after Retrieval is created
# Define axis properties of spectral plot if run_mode == 'evaluate'
retrieval.plotter.spec_xlabel = 'Wavelength [micron]'
retrieval.plotter.spec_ylabel = "Flux [W/m2/micron]"
retrieval.plotter.y_axis_scaling = 1.0
retrieval.plotter.xscale = 'log'
retrieval.plotter.yscale = 'linear'
retrieval.plotter.resolution = 100.0  # maximum resolution, will rebin the data
retrieval.plotter.nsample = 100  # if we want a plot with many sampled spectra

# Define from which observation object to take P-T in evaluation mode, add PT-envelope plotting options
retrieval.plotter.reference_data_name = 'GRAVITY'
retrieval.plotter.temp_limits = [150, 3000]
retrieval.plotter.press_limits = [1e2, 1e-5]

As mentioned at the beginning of this tutorial, this retrieval is very complex and thus takes several days to run, with hundred of cores on a cluster.

To try to run the retrieval anyway, set `run_retrieval` below to `True`, then execute the cells below.

In [10]:
run_retrieval = False

In [ ]:
if run_retrieval:
    retrieval.run(
    sampling_efficiency=0.8,
    const_efficiency_mode=False,
    n_live_points=400,
    log_z_convergence=0.5,
    step_sampler=False,
    warmstart_max_tau=0.5,
    n_iter_before_update=50,
    resume=False,
    max_iters=0,
    frac_remain=0.1,
    l_epsilon=0.3,
    error_checking=True,
    force_serial_error_checking=False,
    seed=12345 # should be -1 for a real retrieval, defined here for reproducibility
    )

Once the retrieval is complete, the easiest way to generate standard output plots is to use the `plot_all` function.

In [12]:
if run_retrieval:
    retrieval.plot_all(contribution=True)

**Contact**

If you need any additional help, don't hesitate to contact [Evert Nasedkin](mailto:nasedkinevert@gmail.com?subject=[petitRADTRANS]%20Retrievals).